In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 读取数据
print("Reading data...")
train_df = pd.read_csv('./data/FBlocation/train.csv')

# 查看数据基本信息
print("Data shape:", train_df.shape)
print("Data columns:", train_df.columns)
print("Data info:")
train_df.info()
print("Data sample:")
display(train_df.head())



Reading data...
Data shape: (29118021, 6)
Data columns: Index(['row_id', 'x', 'y', 'accuracy', 'time', 'place_id'], dtype='object')
Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29118021 entries, 0 to 29118020
Data columns (total 6 columns):
 #   Column    Dtype  
---  ------    -----  
 0   row_id    int64  
 1   x         float64
 2   y         float64
 3   accuracy  int64  
 4   time      int64  
 5   place_id  int64  
dtypes: float64(2), int64(4)
memory usage: 1.3 GB
Data sample:


,row_id,x,y,accuracy,time,place_id
0,0,0.7941,9.0809,54,470702,8523065625
1,1,5.9567,4.7968,13,186555,1757726713
2,2,8.3078,7.0407,74,322648,1137537235
3,3,7.3665,2.5165,65,704587,6567393236
4,4,4.0961,1.1307,31,472130,7440663949


In [2]:
# 准备特征和目标变量
# 处理时间特征
train_df['time_dt'] = pd.to_datetime(train_df['time'], unit='s')
train_df['weekday'] = train_df['time_dt'].dt.weekday
train_df['day'] = train_df['time_dt'].dt.day
train_df['hour'] = train_df['time_dt'].dt.hour
train_df=train_df.query("x > 1.0 &  x < 3 & y > 2.5 & y < 5")
# 查看前5行
train_df.head()


,row_id,x,y,accuracy,time,place_id,time_dt,weekday,day,hour
10,10,2.0173,4.8627,6,21353,8684462954,1970-01-01 05:55:53,3,1,5
31,31,1.5702,2.8070,47,358928,6179383704,1970-01-05 03:42:08,0,5,3
44,44,2.0768,4.4422,53,766037,1367769233,1970-01-09 20:47:17,4,9,20
74,74,1.8616,2.8019,133,769002,4530587605,1970-01-09 21:36:42,4,9,21
94,94,2.9072,4.1222,70,683053,5744032337,1970-01-08 21:44:13,3,8,21


In [3]:

# 选择适合作为特征的列
# 筛选 x 在 [2,5] 范围内的数据
filtered_df=train_df.query("x > 1.5 &  x < 2.5 & y > 2.5 & y < 3")

# 计算每个 place_id 出现的次数
place_counts = filtered_df['place_id'].value_counts()

# 找出出现次数大于3次的 place_id
valid_places = place_counts[place_counts > 3].index

# 进一步筛选数据，只保留出现次数大于3次的 place_id,isin接口跟索引，筛选出place_id在valid_places中的行
filtered_df = filtered_df[filtered_df['place_id'].isin(valid_places)]

# 计算剔除比例
#original_count = train_df[(train_df['x'] >= 0) & (train_df['x'] <= 4) & (train_df['y'] >= 0) & (train_df['y'] <= 4)].shape[0]
original_count = train_df.query("x > 1.5 &  x < 2.5 & y > 2.5 & y < 3").shape[0]
filtered_count = filtered_df.shape[0]
removed_ratio = 1 - (filtered_count / original_count)

print(f"原始数据在指定区域内的记录数: {original_count}")
print(f"剔除出现次数少于等于3次的place_id后的记录数: {filtered_count}")
print(f"剔除比例: {removed_ratio:.4f} ({removed_ratio*100:.2f}%)")
print(f"剩余的不同place_id数量: {filtered_df['place_id'].nunique()}")

原始数据在指定区域内的记录数: 146417
剔除出现次数少于等于3次的place_id后的记录数: 143529
剔除比例: 0.0197 (1.97%)
剩余的不同place_id数量: 1489


In [4]:
filtered_df.head()

,row_id,x,y,accuracy,time,place_id,time_dt,weekday,day,hour
31,31,1.5702,2.8070,47,358928,6179383704,1970-01-05 03:42:08,0,5,3
74,74,1.8616,2.8019,133,769002,4530587605,1970-01-09 21:36:42,4,9,21
159,159,2.4166,2.7762,66,74327,3432339087,1970-01-01 20:38:47,3,1,20
390,390,1.9618,2.6769,989,623985,7419370539,1970-01-08 05:19:45,3,8,5
528,528,2.3636,2.5427,91,684165,7875184665,1970-01-08 22:02:45,3,8,22


In [5]:
import time

filtered_df['dist_from_center'] = (filtered_df['x'] - filtered_df['x'].mean())**2 + (filtered_df['y'] - filtered_df['y'].mean())**2



# 使用过滤后的数据作为特征和目标变量

X = filtered_df[['x', 'y', 'accuracy', 'weekday', 'day', 'hour','dist_from_center']]
y = filtered_df['place_id']

# 分割训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 特征标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# KNN模型构建
print("Training KNN model...")
# 记录开始时间
start_time = time.time()

# 创建KNN分类器，使用默认K=5
knn = KNeighborsClassifier(n_neighbors=5)

# 训练模型
knn.fit(X_train_scaled, y_train)

# 记录结束时间
end_time = time.time()
print(f"KNN模型训练和评估总耗时: {end_time - start_time:.2f} 秒")

# 预测和评估
y_pred = knn.predict(X_val_scaled)
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation accuracy: {accuracy:.4f}")



Training KNN model...
KNN模型训练和评估总耗时: 0.38 秒
Validation accuracy: 0.3720


In [6]:
# 导入GridSearchCV和交叉验证相关工具
from sklearn.model_selection import GridSearchCV

# 尝试不同的K值找到最优模型
param_grid = {
    'n_neighbors': [3,4, 5, 6,7],
    'weights': ['uniform', 'distance']
}

# 记录开始时间
start_time = time.time()

# 创建KNN分类器
knn = KNeighborsClassifier()

# 创建GridSearchCV对象，使用3折交叉验证
grid_search = GridSearchCV(
    estimator=knn,          # 基础模型，这里是KNN分类器
    param_grid=param_grid,  # 参数网格，包含要测试的不同k值和权重选项
    cv=5,                   # 5折交叉验证，将数据分成3份进行验证
    scoring='accuracy',     # 使用准确率作为评估指标
    n_jobs=-1               # 使用所有可用CPU核心加速计算
)

# 拟合网格搜索模型
print("开始网格搜索...")
grid_search.fit(X_train_scaled, y_train)

# 输出最佳参数
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# 记录结束时间
end_time = time.time()
print(f"KNN网格搜索总耗时: {end_time - start_time:.2f} 秒")
print(f"最佳参数: {best_params}")
print(f"交叉验证最佳得分: {best_score:.4f}")


开始网格搜索...


d:\python\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


KNN网格搜索总耗时: 50.66 秒
最佳参数: {'n_neighbors': 7, 'weights': 'distance'}
交叉验证最佳得分: 0.3846


In [7]:

# 使用最佳参数的模型
best_knn = grid_search.best_estimator_
final_accuracy = accuracy_score(y_val, best_knn.predict(X_val_scaled))
print(f"最佳模型验证集准确率: {final_accuracy:.4f}")


最佳模型验证集准确率: 0.3954


In [8]:
# 对最佳KNN模型进行详细评估分析

from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# 获取验证集上的预测结果
y_val_pred = best_knn.predict(X_val_scaled)

In [9]:
y_val_pred

array([4151474164, 3624190747, 5005451423, ..., 8604716564, 2981742738,
       7417026373], dtype=int64)

In [10]:
y_val.values

array([4151474164, 3624190747, 9302589208, ..., 8604716564, 8652491300,
       2495034607], dtype=int64)

In [17]:
# 加载测试数据
print("加载测试数据...")
test_df = pd.read_csv('./data/FBlocation/test.csv',nrows=600000)

# 查看测试数据的前几行
print("测试数据预览:")
print(test_df.head())

# 提取特征
print("准备测试特征...")
X_test = test_df[['x', 'y', 'accuracy', 'time']]

# 如果训练集使用了时间特征，也需要为测试集创建
if 'weekday' in X_train.columns:
    # 转换时间戳为datetime对象
    test_df['time_dt'] = pd.to_datetime(test_df['time'], unit='s')
    

    # 提取时间特征
    test_df['weekday'] = test_df['time_dt'].dt.weekday
    test_df['day'] = test_df['time_dt'].dt.day
    test_df['hour'] = test_df['time_dt'].dt.hour
    test_df['dist_from_center'] = (test_df['x'] - test_df['x'].mean())**2 + (test_df['y'] - test_df['y'].mean())**2
    # 更新测试特征
    X_test = test_df[['x', 'y', 'accuracy', 'weekday', 'day', 'hour','dist_from_center']]

print(f"测试集形状: {X_test.shape}")

# 按训练集标准化测试集
X_test_scaled = scaler.transform(X_test)

加载测试数据...
测试数据预览:
   row_id       x       y  accuracy    time
0       0  0.1675  1.3608       107  930883
1       1  7.3909  2.5301        35  893017
2       2  8.0978  2.3473        62  976933
3       3  0.9990  1.0591        62  907285
4       4  0.6670  9.7254        40  914399
准备测试特征...
测试集形状: (600000, 7)


In [19]:
X_test_scaled.shape

(600000, 7)

In [13]:
X_val_scaled.shape

(28706, 7)

In [18]:

# 使用最佳KNN模型进行预测 #6百万要100分钟
print("使用最佳KNN模型进行预测...")
y_test_pred = best_knn.predict(X_test_scaled)
y_test_pred.shape

使用最佳KNN模型进行预测...


(600000,)

In [ ]:

# 将预测结果添加到测试数据中
test_df['predicted_place_id'] = y_test_pred

# 查看预测结果
print("预测结果预览:")
print(test_df[['row_id', 'x', 'y', 'predicted_place_id']].head())

# 可选：保存预测结果
test_df[['row_id', 'predicted_place_id']].to_csv('knn_predictions.csv', index=False)
print("预测结果已保存到 knn_predictions.csv")
